In [20]:
!pip install pyspark

In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType

In [22]:
spark = SparkSession.builder.appName("BikeAnalysis").getOrCreate()
spark

In [23]:
trips = spark.read.format('csv').option('header', 'true').load("trips.csv")
stations = spark.read.format('csv').option('header', 'true').load("stations.csv")

trips_clean = trips.withColumn("duration", F.col("duration").cast(IntegerType())) \
                   .withColumn("bike_id", F.col("bike_id").cast(IntegerType())) \
                   .withColumn("start_station_id", F.col("start_station_id").cast(IntegerType())) \
                   .withColumn("end_station_id", F.col("end_station_id").cast(IntegerType())) \
                   .withColumn("zip_code", F.col("zip_code"))

stations_clean = stations.withColumn("id", F.col("id").cast(IntegerType())) \
                         .withColumn("lat", F.col("lat").cast(DoubleType())) \
                         .withColumn("long", F.col("long").cast(DoubleType()))

## 1. Найти велосипед с максимальным временем пробега.

In [24]:
trips_valid = trips_clean.filter(F.col("duration").isNotNull())
max_duration_per_bike = (trips_valid
    .groupBy("bike_id")
    .agg(F.max("duration").alias("max_duration"))
    .orderBy(F.col("max_duration").desc()))

max_duration_per_bike.show(1)

+-------+------------+
|bike_id|max_duration|
+-------+------------+
|    535|    17270400|
+-------+------------+
only showing top 1 row


## 2. Найти наибольшее геодезическое расстояние между станциями.

In [25]:
# Делаем Cross Join станций самих с собой, чтобы получить все возможные пары
st_a = stations_clean.selectExpr("id as id_a", "lat as lat_a", "long as long_a")
st_b = stations_clean.selectExpr("id as id_b", "lat as lat_b", "long as long_b")

pairs = st_a.crossJoin(st_b).filter("id_a < id_b") # чтобы не считать расстояние до самой себя и не дублировать пары

# Формула гаверсинуса
R = 6371  # Радиус Земли в км

dist_df = pairs.withColumn("dLat", F.radians(F.col("lat_b") - F.col("lat_a"))) \
               .withColumn("dLon", F.radians(F.col("long_b") - F.col("long_a"))) \
               .withColumn("a",
                    F.sin(F.col("dLat") / 2)**2 +
                    F.cos(F.radians("lat_a")) * F.cos(F.radians("lat_b")) *
                    F.sin(F.col("dLon") / 2)**2
                ) \
               .withColumn("distance", 2 * R * F.atan2(F.sqrt("a"), F.sqrt(1 - F.col("a"))))

max_dist = dist_df.select("id_a", "id_b", "distance").orderBy(F.col("distance").desc())
max_dist.show(1)

+----+----+----------------+
|id_a|id_b|        distance|
+----+----+----------------+
|  16|  60|69.9208759542826|
+----+----+----------------+
only showing top 1 row


## 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [27]:
top_bike_id = max_duration_per_bike.first()["bike_id"]

# Фильтруем все поездки и сортируем их по дате начала
bike_path = trips_clean.filter(F.col("bike_id") == top_bike_id) \
                       .orderBy("start_date") \
                       .select("start_date", "start_station_name", "end_station_name", "duration")

print(f"Путь велосипеда {top_bike_id}:")
bike_path.show(truncate=False)


Путь велосипеда 535:
+---------------+---------------------------------------------+---------------------------------------------+--------+
|start_date     |start_station_name                           |end_station_name                             |duration|
+---------------+---------------------------------------------+---------------------------------------------+--------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |3289    |
|1/1/2014 18:51 |Embarcadero at Sansome                       |Market at 4th                                |1286    |
|1/1/2014 19:48 |Market at 4th                                |South Van Ness at Market                     |795     |
|1/10/2014 20:13|Market at 10th                               |Powell Street BART                           |235     |
|1/10/2014 8:09 |Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |596     |
|1/10/2014 8:21 |San Franci

## 4. Найти количество велосипедов в системе.

In [28]:
unique_bikes_count = trips_clean.select("bike_id").distinct().count()

print(f"Количество велосипедов в системе: {unique_bikes_count}")

Количество велосипедов в системе: 700


## 5. Найти пользователей потративших на поездки более 3 часов.

In [32]:
users_over_3_hours = trips_clean.filter(F.col("zip_code").isNotNull()) \
    .groupBy("zip_code") \
    .agg(F.sum("duration").alias("total_duration")) \
    .filter(F.col("total_duration") > 10800)

users_over_3_hours.show()

+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94102|      19128021|
|   95134|        728023|
|   84606|         95145|
|   80305|        180906|
|   60070|         28919|
|   95519|         30303|
|   43085|         11670|
|   91910|         50488|
|   77339|         13713|
|   48063|         13755|
|   85022|         12682|
|    1090|         20391|
|    2136|         16010|
|   11722|         24331|
|   95138|        155295|
|   94610|       3630628|
|   94404|       3589350|
|   80301|        152189|
|   91326|         65885|
|   90742|         10965|
+--------+--------------+
only showing top 20 rows
